In [30]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()
spark =SparkSession. \
        builder. \
        config('spark.ui.port','0'). \
        config('spark.shuffle.useOldFetchProtocol','true'). \
        config('spark.sql.warehouse.dir',f"/user/{username}/warehouse"). \
        enableHiveSupport(). \
        master('yarn'). \
        getOrCreate()

In [31]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("inferSchema",True) \
.option("header",True) \
.load("/user/itv026162/lendingclub/project/raw/loan_defaulters_data_csv")

In [32]:
loans_def_raw_df

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
ebe2fa66486c78fda...,0.0,0.0,0.0,0.0,0.0,20.860000009100002,33.0,null
f21094e261c23c3a4...,0.0,0.0,1.0,1.0,0.0,0.0,null,93.0
814b80a446d00abd4...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
259a8a3d43849e98e...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
da69229e46e25f2b3...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
785dd25db26ea1874...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
b0171bed5ad8037c2...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
00efd881340dc1344...,0.0,0.0,0.0,0.0,0.0,0.0,60.0,null
9d96fe1a364bcb6f8...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
ce3eac30eacdbf0c4...,0.0,0.0,0.0,0.0,0.0,0.0,null,null


In [33]:
loans_def_raw_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- delinq_2yrs: string (nullable = true)
 |-- delinq_amnt: double (nullable = true)
 |-- pub_rec: string (nullable = true)
 |-- pub_rec_bankruptcies: double (nullable = true)
 |-- inq_last_6mths: string (nullable = true)
 |-- total_rec_late_fee: string (nullable = true)
 |-- mths_since_last_delinq: string (nullable = true)
 |-- mths_since_last_record: string (nullable = true)



In [34]:
loans_def_raw_df.createOrReplaceTempView("loan_def")

In [35]:
spark.sql("select distinct(delinq_2yrs) from loan_def")

delinq_2yrs
1.0
271 monthly payme...
I bike to work on...
183xx
VISA and AMEX cre...
etc. and I feel t...
AZ
017xx
923xx
446xx


In [36]:
spark.sql("select delinq_2yrs, count(*) from loan_def group by delinq_2yrs order by count(*) desc ").show(40)

+------------------+--------+
|       delinq_2yrs|count(1)|
+------------------+--------+
|               0.0| 1838878|
|               1.0|  281335|
|               2.0|   81285|
|               3.0|   29539|
|               4.0|   13179|
|               5.0|    6599|
|               6.0|    3717|
|               7.0|    2062|
|               8.0|    1223|
|               9.0|     818|
|              10.0|     556|
|              11.0|     363|
|              12.0|     264|
|              13.0|     165|
|              14.0|     120|
|              15.0|      87|
|              null|      63|
|              16.0|      55|
|              17.0|      30|
|              18.0|      30|
|              19.0|      23|
|              20.0|      17|
|              21.0|      12|
|                CA|       8|
|                TX|       6|
|debt_consolidation|       5|
|                IL|       5|
|              22.0|       5|
|    small_business|       5|
|                FL|       4|
|         

In [37]:
loan_defaulters_schema = "member_id string,	delinq_2yrs float, delinq_amnt float,	pub_rec float,	pub_rec_bankruptcies float,	inq_last_6mths float,	total_rec_late_fee float,	mths_since_last_delinq float,	mths_since_last_record float"

In [38]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.schema(loan_defaulters_schema) \
.load("/user/itv026162/lendingclub/project/raw/loan_defaulters_data_csv")

In [39]:
loans_def_raw_df.createOrReplaceTempView("loan_def")

In [40]:
spark.sql("select delinq_2yrs, count(*) from loan_def group by delinq_2yrs order by count(*) desc ").show(40)

+-----------+--------+
|delinq_2yrs|count(1)|
+-----------+--------+
|        0.0| 1838878|
|        1.0|  281335|
|        2.0|   81285|
|        3.0|   29539|
|        4.0|   13179|
|        5.0|    6599|
|        6.0|    3717|
|        7.0|    2062|
|        8.0|    1223|
|        9.0|     818|
|       10.0|     556|
|       11.0|     363|
|       12.0|     264|
|       null|     261|
|       13.0|     165|
|       14.0|     120|
|       15.0|      87|
|       16.0|      55|
|       18.0|      30|
|       17.0|      30|
|       19.0|      23|
|       20.0|      17|
|       21.0|      12|
|       22.0|       5|
|       24.0|       4|
|       26.0|       3|
|       23.0|       2|
|       25.0|       2|
|       29.0|       2|
|       3.44|       2|
|       30.0|       2|
|       1.41|       1|
|      22.62|       1|
|      26.24|       1|
|      17.18|       1|
|      17.17|       1|
|      13.76|       1|
|       5.52|       1|
|       9.44|       1|
|      22.95|       1|
+----------

In [41]:
from pyspark.sql.functions import col

In [42]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs",col("delinq_2yrs").cast("integer")).fillna(0, subset = ["delinq_2yrs"])

In [43]:
loans_def_processed_df.createOrReplaceTempView("loan_def")

In [44]:
spark.sql("select count(*) from loan_def where delinq_2yrs is null ")

count(1)
0


In [45]:
spark.sql("select * from loan_def where delinq_2yrs = 0 ")

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
ebe2fa66486c78fda...,0,0.0,0.0,0.0,0.0,20.86,33.0,null
f21094e261c23c3a4...,0,0.0,1.0,1.0,0.0,0.0,null,93.0
814b80a446d00abd4...,0,0.0,0.0,0.0,0.0,0.0,null,null
259a8a3d43849e98e...,0,0.0,0.0,0.0,0.0,0.0,null,null
da69229e46e25f2b3...,0,0.0,0.0,0.0,0.0,0.0,null,null
785dd25db26ea1874...,0,0.0,0.0,0.0,0.0,0.0,null,null
b0171bed5ad8037c2...,0,0.0,0.0,0.0,0.0,0.0,null,null
00efd881340dc1344...,0,0.0,0.0,0.0,0.0,0.0,60.0,null
9d96fe1a364bcb6f8...,0,0.0,0.0,0.0,0.0,0.0,null,null
ce3eac30eacdbf0c4...,0,0.0,0.0,0.0,0.0,0.0,null,null


In [46]:
loans_def_deling_df = spark.sql("select member_id, delinq_2yrs, delinq_amnt, int(mths_since_last_delinq) from loan_def where delinq_2yrs > 0 or mths_since_last_delinq > 0 ")

In [47]:
loans_def_deling_df.count()

1106163

In [48]:
loans_def_records_eng_df = spark.sql("select member_id from loan_def where pub_rec > 0.0 or pub_rec_bankruptcies > 0.0 or inq_last_6mths > 0.0 ")

In [49]:
loans_def_records_eng_df.count()

1070125

In [50]:
loans_def_deling_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned/loan_defaulters_deling_csv") \
.save()

In [51]:
loans_def_records_eng_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned/loan_defaulters_records_enq_csv") \
.save()

In [52]:
spark.stop()